# Zajęcia 4 — cechy i potok przetwarzania

Notebook pokazuje, jak jedna transformacja cech chroni przed rozjazdem treningu i predykcji, oraz jak zbudować potok z jawnymi zależnościami, pamięcią podręczną i obsługą awarii.

Komórki opisane jako **zadanie** wypełniasz samodzielnie. Komórka pod nimi zawiera rozwiązanie ukryte magią `%%skip True` — aby je wykonać, zmień nagłówek na `%%skip False`.

## 1. Przygotowanie notebooka

Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
import hashlib
import json
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable

import joblib
import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_iris
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "skip_kernel_extension.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

BASE_DIR = REPO_ROOT / "artifacts" / "zajecia_04"
for sub in ("data", "models", "reports"):
    (BASE_DIR / sub).mkdir(parents=True, exist_ok=True)

DATA_PATH = BASE_DIR / "data" / "iris.csv"
TRAIN_PATH = BASE_DIR / "data" / "train.csv"
TEST_PATH = BASE_DIR / "data" / "test.csv"
MODEL_PATH = BASE_DIR / "models" / "model.joblib"
VALIDATION_PATH = BASE_DIR / "reports" / "validation.json"
METRICS_PATH = BASE_DIR / "reports" / "metrics.json"
CACHE_PATH = BASE_DIR / ".pipeline_cache.json"

FEATURE_COLUMNS = [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)",
]
TARGET_COLUMN = "target"

load_iris(as_frame=True).frame.to_csv(DATA_PATH, index=False)
frame = pd.read_csv(DATA_PATH)
print(f"Dane: {DATA_PATH.relative_to(REPO_ROOT)} ({len(frame)} wierszy)")

%load_ext skip_kernel_extension

## 2. Wspólna transformacja cech

Transformacja ma być **jednym obiektem**, wersjonowanym razem z modelem. `ColumnTransformer` wybiera kolumny po nazwie, a nie po pozycji — to pierwsza warstwa ochrony przed rozjazdem.

**Zadanie:** napisz `build_feature_pipeline()` zwracającą `ColumnTransformer` skalujący kolumny z `FEATURE_COLUMNS` oraz `build_model(max_iter, seed)` łączącą ją z regresją logistyczną.

In [ ]:
# Zadanie: zbuduj wspólną transformację cech i model.


In [ ]:
%%skip True
def build_feature_pipeline() -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[("numeric", StandardScaler(), FEATURE_COLUMNS)],
        remainder="drop",
    )


def build_model(max_iter: int = 500, seed: int = 42) -> Pipeline:
    return Pipeline(
        steps=[
            ("features", build_feature_pipeline()),
            ("classifier", LogisticRegression(max_iter=max_iter, random_state=seed)),
        ]
    )


safe_model = build_model()
safe_model.fit(frame[FEATURE_COLUMNS], frame[TARGET_COLUMN])
print(safe_model)

## 3. Ścieżka predykcyjna i test zgodności

Model wytrenowaliśmy na tabeli, w której cechy mają jawne nazwy i kolejność. W produkcji często przychodzi jednak pojedyncze żądanie HTTP z rekordem JSON, na przykład `{"petal width (cm)": 0.2, "sepal length (cm)": 5.1, ...}`. To osobna ścieżka przetwarzania danych: zanim model przewidzi klasę, aplikacja musi zamienić rekord na wejście zgodne z tym, na czym model był trenowany.

### Jaki problem rozwiązujemy?

Wartości w rekordzie mają znaczenie, ale równie ważne jest to, **do której cechy należy każda wartość**. Jeśli kod po prostu zbierze wartości słownika do tablicy, zmiana kolejności kluczy może podać długość płatka w miejsce długości działki. Model najczęściej nie zgłosi błędu — dostanie poprawną liczbę, lecz przypisaną do niewłaściwej cechy, i zwróci wiarygodnie wyglądającą, ale błędną predykcję.

Adapter `to_feature_frame(records)` ma być granicą kontraktu wejściowego. Dla każdego rekordu powinien:

- wykryć brak wymaganej cechy i przerwać przetwarzanie z czytelnym `ValueError`, zamiast po cichu tworzyć niekompletne wejście;
- ułożyć kolumny dokładnie w kolejności `FEATURE_COLUMNS`, niezależnie od kolejności kluczy w JSON;
- zwrócić wartości w postaci numerycznej oczekiwanej przez potok modelu.

### Jak to wygląda w praktyce?

- **Różni klienci API:** aplikacja webowa, usługa w Javie i skrypt w Pythonie wysyłają te same pola, ale serializują klucze w różnej kolejności. Jeśli serwer polega na kolejności wartości, wynik zależy od klienta.
- **Zmiana payloadu:** klient dodaje pola takie jak `request_id` albo `source`, lub pomija pole cechy, które uznaje za opcjonalne. Adapter powinien ignorować pola spoza kontraktu i odrzucić rekord bez wymaganej cechy, zamiast przesunąć wartości.
- **Dwie drogi do tego samego modelu:** zadanie wsadowe przewiduje klasy dla całego pliku, a endpoint obsługuje pojedyncze żądania. Różne przygotowanie cech może sprawić, że ten sam rekord dostanie inną predykcję zależnie od drogi.

### Co ma wykazać test?

Test porównuje predykcje dla tych samych obserwacji: raz przekazanych razem jako ramka wsadowa, a raz jako lista rekordów. Zgodność potwierdza, że adapter rekordów nie zmienił znaczenia cech. Osobna próba z brakującym polem sprawdza, czy kontrakt odrzuca niekompletne wejście. Następny punkt kontrolny pokazuje celowo naiwną implementację i to, jak kolejność kluczy może ją uszkodzić.

**Zadanie:** napisz `to_feature_frame(records)`, a następnie funkcję przewidującą klasy dla rekordów i test porównujący jej wyniki z predykcjami wsadowymi. Wartości obecności i kolejności kolumn to nie pełna walidacja danych: zakresy, jednostki i sens wartości wymagają osobnych reguł.


In [ ]:
# Zadanie: ścieżka predykcyjna z kontraktem oraz test zgodności obu ścieżek.


In [ ]:
%%skip True
def to_feature_frame(records: list[dict]) -> pd.DataFrame:
    candidate = pd.DataFrame.from_records(records)
    missing = sorted(set(FEATURE_COLUMNS) - set(candidate.columns))
    if missing:
        raise ValueError(f"Brak wymaganych cech: {missing}")
    return candidate[FEATURE_COLUMNS].astype(float)


def predict_records(model, records: list[dict]) -> np.ndarray:
    return model.predict(to_feature_frame(records))


def assert_paths_agree(model, batch: pd.DataFrame) -> None:
    records = batch[FEATURE_COLUMNS].to_dict(orient="records")
    batch_predictions = model.predict(batch[FEATURE_COLUMNS])
    record_predictions = predict_records(model, records)
    if not np.array_equal(batch_predictions, record_predictions):
        raise AssertionError("Ścieżka wsadowa i rekordowa dają różne predykcje.")
    print(f"Obie ścieżki zgodne dla {len(batch)} rekordów.")


assert_paths_agree(safe_model, frame)

try:
    predict_records(safe_model, [{"sepal length (cm)": 5.1}])
except ValueError as error:
    print(f"Kontrakt wejścia zadziałał: {error}")

## 4. Punkt kontrolny 1 — rozjazd kolejności kolumn

Pokazujemy implementację, którą łatwo napisać przypadkiem: rekord zamieniany na tablicę liczb w kolejności kluczy. Model przyjmuje ją bez protestu, bo tablica nie niesie nazw kolumn.

**Do opisania:** dlaczego model nie zgłosił błędu i który mechanizm wykrył problem?

In [ ]:
positional_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=500, random_state=42)),
    ]
)
positional_model.fit(frame[FEATURE_COLUMNS].to_numpy(), frame[TARGET_COLUMN])


def naive_predict(model, records: list[dict]) -> np.ndarray:
    matrix = np.asarray([[float(value) for value in record.values()] for record in records])
    return model.predict(matrix)


ordered = frame[FEATURE_COLUMNS].to_dict(orient="records")
shuffled_keys = list(reversed(FEATURE_COLUMNS))
shuffled = [{key: record[key] for key in shuffled_keys} for record in ordered]

reference = naive_predict(positional_model, ordered)
damaged = naive_predict(positional_model, shuffled)

print(f"Zgodność predykcji po zmianie kolejności kluczy: {np.mean(reference == damaged):.2%}")
print("Model zgłosił błąd: nie — otrzymał tablicę liczb bez nazw kolumn.\n")

safe_reference = predict_records(safe_model, ordered)
safe_damaged = predict_records(safe_model, shuffled)
print(f"Ścieżka z kontraktem, zgodność: {np.mean(safe_reference == safe_damaged):.2%}")

## 5. Wyciek przez przetwarzanie

Selekcja cech, skalowanie i imputacja używają statystyk policzonych na danych. Wykonane przed podziałem zbioru przenoszą informację z testu do treningu.

Demonstracja jest skrajna, ale realistyczna: **dane całkowicie losowe**, bez żadnego związku między cechami a etykietą. Poprawnie zbudowany potok powinien dać wynik na poziomie losowania.

**Zadanie:** porównaj dwa warianty — selekcję cech wykonaną przed walidacją krzyżową oraz selekcję umieszczoną wewnątrz `Pipeline`. Użyj `cross_val_score` z `cv=5`.

In [ ]:
# Tworzy powtarzalne dane testowe: 200 próbek z 2000 cechami oraz losowe etykiety binarne.
# rng to generator liczb losowych z ustalonym ziarnem (seed=0)
rng = np.random.default_rng(0)
noise_x = rng.normal(size=(200, 2000))
noise_y = rng.integers(0, 2, size=200)

# Zadanie: porównaj selekcję cech przed podziałem i wewnątrz potoku.


In [ ]:
%%skip True

### Wybór przed podziałem na test/train
selected = SelectKBest(f_classif, k=10).fit_transform(noise_x, noise_y) # spośród 2000 kolumn wybierz 10, które statystycznie najlepiej pasują do etykiet.
leaky_score = cross_val_score(
    LogisticRegression(max_iter=1000), selected, noise_y, cv=5
).mean()

### Wybór po podziale na test/train (wewnątrz potoku)
honest_pipeline = Pipeline(
    steps=[
        ("select", SelectKBest(f_classif, k=10)), # spośród 2000 kolumn wybierz 10, które statystycznie najlepiej pasują do etykiet.
        ("classifier", LogisticRegression(max_iter=1000)),
    ]
)
honest_score = cross_val_score(honest_pipeline, noise_x, noise_y, cv=5).mean()

print(f"Selekcja przed podziałem (wyciek): {leaky_score:.3f}")
print(f"Selekcja wewnątrz potoku:          {honest_score:.3f}")
print("\nDane są losowe, więc uczciwy wynik powinien być bliski 0.5.")

## 6. Poprawność czasowa

Gdy problem ma wymiar czasu, losowy podział zbioru ukrywa zmianę reżimu. Generujemy dane, w których w ostatnich 25% obserwacji odwraca się zależność między cechą a etykietą.

**Zadanie:** porównaj accuracy przy podziale losowym i przy podziale czasowym (ostatnie 25% obserwacji jako zbiór testowy).

In [ ]:
# Tworzymy 1200 obserwacji i generator liczb losowych z ustalonym ziarnem.
# Losujemy dwie cechy o rozkładzie normalnym.
# Wpływ feature_a jest dodatni dla pierwszych 900 obserwacji, a potem zmienia znak.
# Obliczamy wynik logitowy z uwzględnieniem obu cech.
# Zamieniamy logit na prawdopodobieństwo i losujemy etykiety 0 lub 1.
steps = 1200
rng = np.random.default_rng(7)
feature_a = rng.normal(size=steps)
feature_b = rng.normal(size=steps)
coefficient = np.where(np.arange(steps) < 900, 1.0, -1.0)
logit = coefficient * 2.0 * feature_a + 0.5 * feature_b
label = (rng.random(steps) < 1.0 / (1.0 + np.exp(-logit))).astype(int)

timed = pd.DataFrame({"t": np.arange(steps), "a": feature_a, "b": feature_b, "y": label})

# Zadanie: porównaj podział losowy i czasowy.


Poniższy kod oblicza **wynik `logit`**, który określa, jak mocno cechy przemawiają za etykietą `1`:

```python
logit = coefficient * 2.0 * feature_a + 0.5 * feature_b
```

- `feature_a` ma silny wpływ — jest mnożona przez `2.0`.
- `coefficient` wynosi `1` dla pierwszych 900 obserwacji, a potem `-1`. Zmienia więc kierunek wpływu `feature_a`: najpierw większa wartość tej cechy sprzyja etykiecie `1`, a po 900. obserwacji — etykiecie `0`.
- `feature_b` ma słabszy wpływ, bo jest mnożona przez `0.5`.
- `logit` to jeszcze nie prawdopodobieństwo. Może przyjmować dowolne wartości, także ujemne.

Poniższy kod zamienia ten wynik na prawdopodobieństwo i losuje etykietę:

```python
label = (rng.random(steps) < 1.0 / (1.0 + np.exp(-logit))).astype(int)
```

Wyrażenie

```python
1.0 / (1.0 + np.exp(-logit))
```

to funkcja sigmoid, która zamienia `logit` na liczbę od 0 do 1, czyli prawdopodobieństwo etykiety `1`.

Przykładowo:

| `logit` | Prawdopodobieństwo etykiety `1` |
|---:|---:|
| `2` | ok. `0.88` |
| `0` | `0.50` |
| `-2` | ok. `0.12` |

`rng.random(steps)` losuje dla każdej obserwacji liczbę od 0 do 1. Jeśli ta liczba jest mniejsza od obliczonego prawdopodobieństwa, etykieta zostaje ustawiona na `True`, czyli po `.astype(int)` na `1`. W przeciwnym razie wynosi `0`.

Na przykład, gdy prawdopodobieństwo to `0.88`:
- los `0.70` daje etykietę `1`, bo `0.70 < 0.88`;
- los `0.95` daje etykietę `0`, bo `0.95` nie jest mniejsze od `0.88`.

W ten sposób etykiety są losowe, ale ich prawdopodobieństwo zależy od cech. Po zmianie `coefficient` znak wpływu `feature_a` się odwraca.

In [ ]:
%%skip True
features = timed[["a", "b"]]
targets = timed["y"]

x_train, x_test, y_train, y_test = train_test_split(
    features, targets, test_size=0.25, random_state=42, shuffle=True
)
random_split = accuracy_score(
    y_test, LogisticRegression().fit(x_train, y_train).predict(x_test)
)

cut = int(len(timed) * 0.75)
time_split = accuracy_score(
    targets.iloc[cut:],
    LogisticRegression().fit(features.iloc[:cut], targets.iloc[:cut]).predict(features.iloc[cut:]),
)

print(f"Podział losowy:  {random_split:.3f}")
print(f"Podział czasowy: {time_split:.3f}")
print("\nPodział losowy wpuszcza do treningu obserwacje z okresu po zmianie reżimu.")

## 7. Potok jako graf etapów

Etap deklaruje, co czyta (`deps`), co produkuje (`outs`) i od jakich parametrów zależy. Z tych deklaracji wynika kolejność wykonania, pamięć podręczna i lineage.

Odcisk etapu liczymy z sum kontrolnych wejść i wartości parametrów — dokładnie tak, jak robi to DVC. Ta komórka jest gotowa — uruchom ją bez zmian.

### Przygotowanie prostego runnera potoku

Ta komórka definiuje mechanizm do uruchamiania etapów przetwarzania danych. Runner zapamiętuje, na podstawie jakich plików wejściowych i parametrów uruchomiono każdy etap. Przy kolejnym uruchomieniu może pominąć etap, jeśli jego dane wejściowe się nie zmieniły, a pliki wynikowe nadal istnieją.

#### Co opisuje `Stage`?

`Stage` to struktura przechowująca informacje o jednym etapie:

- `name` — nazwa etapu;
- `deps` — pliki wejściowe, od których zależy;
- `outs` — oczekiwane pliki wynikowe;
- `params` — dodatkowe parametry wpływające na wynik;
- `run` — funkcja wykonująca etap.

Utworzenie obiektu `Stage` opisuje etap, ale go jeszcze nie uruchamia.

#### Jak wykrywane są zmiany?

`file_digest(path)` oblicza skrót SHA-256 zawartości pliku. Czyta plik fragmentami po 65 536 bajtów, więc nie musi wczytywać całego dużego pliku naraz. Jeśli plik nie istnieje, funkcja zwraca napis `"MISSING"`.

`fingerprint(stage)` tworzy skrót opisujący aktualny stan etapu. Uwzględnia jego nazwę, parametry oraz skróty plików wejściowych. Jeśli któryś z tych elementów się zmieni, zmieni się też skrót etapu.

#### Co robi `run_pipeline(stages)`?

Funkcja przechodzi po etapach w podanej kolejności:

1. Odczytuje zapisane wcześniej skróty z pliku cache albo zaczyna od pustego cache.
2. Oblicza aktualny skrót etapu.
3. Sprawdza, czy wszystkie pliki wynikowe istnieją.
4. Pomija etap, jeśli jego skrót jest taki sam jak zapisany w cache i wszystkie wyniki istnieją.
5. W przeciwnym razie wywołuje `stage.run()`, zapisuje aktualny skrót do cache i oznacza etap jako wykonany.
6. Zwraca tabelę z nazwą etapu, jego statusem i skrótem.

Cache pozwala zapamiętać stan między kolejnymi uruchomieniami notebooka. `print` na końcu tylko potwierdza, że definicje runnera zostały wykonane — nie uruchamia żadnego etapu.

#### Jak użyć runnera?

Najpierw definiuje się funkcje wykonujące pracę, a następnie tworzy etapy i przekazuje je do `run_pipeline`:

```python
etapy = [
    Stage(
        name="przygotowanie_danych",
        deps=[plik_wejsciowy],
        outs=[plik_wynikowy],
        params={"wersja": 1},
        run=przygotuj_dane,
    )
]

raport = run_pipeline(etapy)
raport

In [ ]:
@dataclass
class Stage:
    name: str
    deps: list
    outs: list
    params: dict = field(default_factory=dict)
    run: Callable[[], None] = None


def file_digest(path: Path) -> str:
    if not path.exists():
        return "MISSING"
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(65536), b""):
            digest.update(chunk)
    return digest.hexdigest()


def fingerprint(stage: Stage) -> str:
    parts = [stage.name, json.dumps(stage.params, sort_keys=True)]
    parts += [file_digest(dep) for dep in stage.deps]
    return hashlib.sha256("|".join(parts).encode("utf-8")).hexdigest()[:16]


def run_pipeline(stages: list) -> pd.DataFrame:
    cache = json.loads(CACHE_PATH.read_text(encoding="utf-8")) if CACHE_PATH.exists() else {}
    log = []
    for stage in stages:
        current = fingerprint(stage)
        outputs_present = all(out.exists() for out in stage.outs)
        if cache.get(stage.name) == current and outputs_present:
            log.append({"etap": stage.name, "status": "pominięty (cache)", "odcisk": current})
            continue
        stage.run()
        cache[stage.name] = fingerprint(stage)
        CACHE_PATH.write_text(json.dumps(cache, indent=2), encoding="utf-8")
        log.append({"etap": stage.name, "status": "wykonany", "odcisk": cache[stage.name]})
    return pd.DataFrame(log)


print("Runner potoku gotowy.")

## 8. Definicja etapów

Budujemy cztery etapy: `validate → featurize → train → evaluate`. Zwróć uwagę, że wyjście jednego etapu jest wejściem następnego — to właśnie tworzy graf.

**Zadanie:** dopisz etapy `featurize`, `train` i `evaluate` wraz z ich zależnościami i parametrami.

In [ ]:
# Zadanie: dokończ definicję etapów potoku.
def build_stages(params: dict) -> list:
    def validate():
        data = pd.read_csv(DATA_PATH)
        report = {
            "rows": int(len(data)),
            "columns": list(data.columns),
            "missing_values": int(data.isna().sum().sum()),
        }
        VALIDATION_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")

    return [
        Stage("validate", deps=[DATA_PATH], outs=[VALIDATION_PATH], run=validate),
    ]


### Prosty runner potoku

Ten kod przygotowuje mały runner, czyli mechanizm do uruchamiania kolejnych etapów przetwarzania danych. Jego celem jest uniknięcie ponownego wykonywania etapów, których dane wejściowe się nie zmieniły i których wyniki nadal istnieją.

#### Co opisuje `Stage`?

`Stage` przechowuje informacje o jednym etapie:

- `name` — nazwę etapu,
- `deps` — pliki wejściowe, od których etap zależy,
- `outs` — pliki, które etap ma utworzyć,
- `params` — parametry wpływające na jego działanie,
- `run` — funkcję wykonującą etap.

Samo utworzenie obiektu `Stage` nie uruchamia pracy. Opisuje tylko, co dany etap powinien zrobić.

#### Jak runner wykrywa zmiany?

`file_digest` oblicza skrót SHA-256 zawartości pliku. Dzięki temu można wykryć, że plik wejściowy zmienił się, nawet jeśli nadal ma tę samą nazwę. Dla nieistniejącego pliku zwraca `"MISSING"`.

`fingerprint` tworzy skrót etapu na podstawie jego nazwy, parametrów i zawartości plików wejściowych. Runner porównuje ten skrót z zapisanym wcześniej w cache.

#### Co robi `run_pipeline`?

Dla każdego etapu runner:

1. oblicza jego aktualny skrót;
2. sprawdza, czy wszystkie pliki wynikowe istnieją;
3. pomija etap, jeśli jego skrót się nie zmienił, a wyniki istnieją;
4. w przeciwnym razie uruchamia `stage.run()` i zapisuje nowy skrót w cache;
5. zwraca tabelę z informacją, które etapy wykonano, a które pominięto.

Cache to mały plik z zapamiętanymi skrótami. Dzięki niemu przy kolejnym uruchomieniu runner może rozpoznać, czy trzeba powtórzyć pracę.

#### Jak tego użyć?

Najpierw tworzy się funkcje wykonujące poszczególne etapy, a potem opisuje etapy i przekazuje je do runnera:

```python
etapy = [
    Stage(
        name="przygotowanie_danych",
        deps=[sciezka_do_surowych_danych],
        outs=[sciezka_do_danych_po_przygotowaniu],
        params={"wersja": 1},
        run=funkcja_przygotowujaca_dane,
    )
]

raport = run_pipeline(etapy)

In [ ]:
%%skip True
def build_stages(params: dict) -> list:
    def validate():
        data = pd.read_csv(DATA_PATH)
        report = {
            "rows": int(len(data)),
            "columns": list(data.columns),
            "missing_values": int(data.isna().sum().sum()),
        }
        VALIDATION_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")

    def featurize():
        data = pd.read_csv(DATA_PATH)
        train_part, test_part = train_test_split(
            data,
            test_size=params["test_size"],
            random_state=params["seed"],
            stratify=data[TARGET_COLUMN],
        )
        train_part.to_csv(TRAIN_PATH, index=False)
        test_part.to_csv(TEST_PATH, index=False)

    def train():
        if params.get("fail_train"):
            raise RuntimeError("Symulowana awaria etapu treningu.")
        data = pd.read_csv(TRAIN_PATH)
        model = build_model(max_iter=params["max_iter"], seed=params["seed"])
        model.fit(data[FEATURE_COLUMNS], data[TARGET_COLUMN])
        joblib.dump(model, MODEL_PATH)

    def evaluate():
        data = pd.read_csv(TEST_PATH)
        model = joblib.load(MODEL_PATH)
        predictions = model.predict(data[FEATURE_COLUMNS])
        metrics = {
            "accuracy": round(float(accuracy_score(data[TARGET_COLUMN], predictions)), 4),
            "f1_macro": round(
                float(f1_score(data[TARGET_COLUMN], predictions, average="macro")), 4
            ),
        }
        METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

    return [
        Stage("validate", deps=[DATA_PATH], outs=[VALIDATION_PATH], run=validate),
        Stage(
            "featurize",
            deps=[DATA_PATH],
            outs=[TRAIN_PATH, TEST_PATH],
            params={"test_size": params["test_size"], "seed": params["seed"]},
            run=featurize,
        ),
        Stage(
            "train",
            deps=[TRAIN_PATH],
            outs=[MODEL_PATH],
            params={"max_iter": params["max_iter"], "seed": params["seed"]},
            run=train,
        ),
        Stage("evaluate", deps=[TEST_PATH, MODEL_PATH], outs=[METRICS_PATH], run=evaluate),
    ]


PARAMS = {"test_size": 0.2, "seed": 42, "max_iter": 500}
print([stage.name for stage in build_stages(PARAMS)])

## 9. Pierwsze uruchomienie i pamięć podręczna

Uruchamiamy potok dwa razy z rzędu. Za drugim razem nic się nie zmieniło, więc żaden etap nie powinien być liczony ponownie. Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
CACHE_PATH.unlink(missing_ok=True)

print("Przebieg 1 (od zera):")
display(run_pipeline(build_stages(PARAMS)))

print("Przebieg 2 (bez zmian):")
display(run_pipeline(build_stages(PARAMS)))

print(json.loads(METRICS_PATH.read_text(encoding="utf-8")))

## 10. Punkt kontrolny 2 — co unieważnia wynik etapu

Porównujemy dwie sytuacje: zmianę parametru dotyczącego jednego etapu oraz zmianę pliku wejściowego.

**Do opisania:** dlaczego zmiana `max_iter` nie unieważniła etapu `featurize`, a zmiana pliku danych unieważniła wszystko?

In [ ]:
print("Zmiana parametru max_iter (dotyczy tylko etapu train):")
display(run_pipeline(build_stages({**PARAMS, "max_iter": 1000})))

modified = pd.read_csv(DATA_PATH)
modified.loc[0, "sepal length (cm)"] = 5.2
modified.to_csv(DATA_PATH, index=False)

print("Zmiana pliku danych wejściowych:")
display(run_pipeline(build_stages({**PARAMS, "max_iter": 1000})))

## 11. Punkt kontrolny 3 — awaria etapu

Symulujemy awarię treningu. Sprawdzamy, co pozostało aktualne i co trzeba odtworzyć po naprawie.

**Do opisania:** które artefakty przetrwały awarię i dlaczego etap `evaluate` w ogóle się nie uruchomił?

In [ ]:
MODEL_PATH.unlink(missing_ok=True)
METRICS_PATH.unlink(missing_ok=True)

try:
    run_pipeline(build_stages({**PARAMS, "fail_train": True}))
except RuntimeError as error:
    print(f"Potok zatrzymany: {error}\n")

print(f"validation.json istnieje: {VALIDATION_PATH.exists()}")
print(f"train.csv istnieje:       {TRAIN_PATH.exists()}")
print(f"model.joblib istnieje:    {MODEL_PATH.exists()}")
print(f"metrics.json istnieje:    {METRICS_PATH.exists()}\n")

print("Po naprawie:")
display(run_pipeline(build_stages(PARAMS)))

## 12. Potok w narzędziu — `dvc.yaml`

Napisana przed chwilą pamięć podręczna działa, ale jest naszym własnym kodem, który trzeba utrzymywać. Dokładnie tę samą logikę realizuje DVC — z tą różnicą, że rozumie też parametry, artefakty binarne i magazyn zdalny.

Budujemy więc ten sam potok jeszcze raz, tym razem jako **prawdziwy projekt na dysku**:

```text
prepare  →  validate  →  split  →  train  →  evaluate
```

Zwróć uwagę na krawędź `validate → split`: etap podziału zależy od **raportu walidacji**, a nie tylko od danych. Dzięki temu kontrakt danych z zajęć 3 przestaje być osobnym rytuałem i staje się bramką, przez którą potok musi przejść.

**Zadanie:** przygotuj katalog potoku, plik `params.yaml`, skrypty pięciu etapów oraz `dvc.yaml` z zadeklarowanymi `deps`, `outs`, `params` i `metrics`.

In [ ]:
# Zadanie: zbuduj projekt potoku i opisz go w dvc.yaml.
POTOK = BASE_DIR / "potok"
for sub in ("src", "data", "models", "reports"):
    (POTOK / sub).mkdir(parents=True, exist_ok=True)
print(f"Katalog potoku: {POTOK.relative_to(REPO_ROOT)}")


## Utworzenie przykładowego potoku DVC

Ta komórka **tworzy na dysku mały projekt przetwarzania danych**. Zapisuje pliki z kodem, parametrami i opisem etapów, a następnie inicjalizuje DVC w katalogu projektu. Sama definicja etapów nie oznacza jeszcze, że cały potok został wykonany.

Przykład wykorzystuje zbiór Iris: dane są przygotowywane, sprawdzane, dzielone na część treningową i testową, a następnie używane do trenowania i oceny modelu.

### Jakie pliki powstają?

Po wykonaniu tej komórki w katalogu `POTOK` powstają pliki konfiguracyjne i skrypty:

| Plik | Do czego służy |
|---|---|
| `params.yaml` | Przechowuje parametry podziału danych i modelu, np. rozmiar zbioru testowego i liczbę iteracji. |
| `src/prepare.py` | Pobiera przykładowy zbiór Iris z biblioteki scikit-learn i zapisuje go do pliku CSV. |
| `src/validate.py` | Sprawdza, czy dane mają oczekiwane kolumny i wartości; zapisuje raport walidacji. |
| `src/split.py` | Dzieli dane na zbiory treningowy i testowy. |
| `src/train.py` | Uczy model regresji logistycznej i zapisuje go na dysku. |
| `src/evaluate.py` | Ocenia model na zbiorze testowym i zapisuje metryki. |
| `dvc.yaml` | Opisuje etapy potoku, ich polecenia, zależności, parametry i wyniki. |
| `.dvc/` | Zawiera konfigurację DVC. DVC tworzy ją dzięki poleceniu `dvc init`. |

Pliki z wynikami — takie jak `data/iris.csv`, `data/train.csv`, `models/model.joblib` czy `reports/metrics.json` — powstaną dopiero po uruchomieniu potoku. Podobnie `dvc.lock` jest generowany przez DVC po wykonaniu etapów; zapisuje wtedy m.in. informacje o użytych parametrach i wersjach danych wejściowych.

DVC może przechowywać kopie wyników w lokalnym cache, aby móc je ponownie wykorzystać. W tej komórce inicjalizacja odbywa się z opcją `--no-scm`, czyli bez integracji z Git. To wygodne w demonstracji, ale projekt nie uzyskuje w ten sposób historii wersji w Git ani zdalnego magazynu DVC.

### Co to jest `dvc.yaml`?

`dvc.yaml` to **deklaratywny opis potoku danych**. Zamiast ręcznie uruchamiać skrypty w odpowiedniej kolejności, opisujemy w nim:

- jakie etapy składają się na potok;
- jakie polecenie wykonuje każdy etap;
- od jakich plików i parametrów etap zależy;
- jakie pliki tworzy;
- które pliki są metrykami lub wykresami.

DVC analizuje te informacje i buduje graf zależności. Polecenie:

```bash
python -m dvc repro
```

uruchamia potrzebne etapy we właściwej kolejności. Jeśli zależności i parametry etapu się nie zmieniły, a jego wyniki są dostępne, DVC może pominąć ponowne wykonanie tego etapu. Jeśli zmieni się wejście, parametr albo zależny skrypt, DVC może ponownie wykonać etap i etapy, które od niego zależą.

Kolejność etapów w pliku YAML nie jest tym, co przede wszystkim określa kolejność wykonania. Ważniejsze są zależności między plikami: wynik jednego etapu może być wejściem kolejnego.

### Jak używać projektu po utworzeniu?

Polecenia trzeba uruchomić w katalogu projektu `POTOK`. W notebooku katalog ten można ustawić jako bieżący katalog albo podać jego pełną ścieżkę w terminalu.

```bash
python -m dvc dag
```

Wyświetla graf zależności etapów.

```bash
python -m dvc repro
```

Uruchamia potok lub tylko te etapy, których wykonanie jest potrzebne.

```bash
python -m dvc status
```

Pokazuje, czy wejścia lub etapy różnią się od stanu zapisanego przez DVC.

```bash
python -m dvc metrics show
```

Wyświetla zapisane metryki, w tym `accuracy` i `f1_macro`.

`python -m dvc` uruchamia DVC przez interpreter Pythona bieżącego środowiska. W tej komórce zmienna środowiskowa `DVC_ENV` dodatkowo ustawia `PATH` tak, aby polecenia `python` z `dvc.yaml` wskazywały na to samo środowisko.

Aby polecenia zadziałały, środowisko musi mieć zainstalowane wymagane biblioteki, takie jak DVC, pandas, scikit-learn, PyYAML, Pandera i joblib. Sam DVC nie instaluje zależności Pythona zdefiniowanych w skryptach.

### Co jeszcze można opisać w `dvc.yaml`?

Poza elementami pokazanymi w tym przykładzie, DVC obsługuje m.in.:

- `plots` — pliki wykresów, które DVC może porównywać;
- `wdir` — katalog roboczy, z którego uruchamiane jest polecenie etapu;
- `vars` — wartości konfiguracyjne używane w pliku YAML;
- `foreach` i `matrix` — generowanie wielu podobnych etapów, np. dla różnych zestawów parametrów;
- `always_changed` — wymuszenie uruchamiania etapu przy każdym `dvc repro`;
- `frozen` — oznaczenie etapu jako niezmienianego automatycznie;
- konfigurację wyjść, np. czy wynik ma być przechowywany w cache DVC.

Te możliwości są przydatne w większych projektach, ale nie są konieczne w tym demonstracyjnym potoku.

## Szczegółowe objaśnienie `dvc.yaml` z tej komórki

Poniżej omówione są wszystkie wpisy w konkretnym pliku tworzonym przez komórkę.

### `stages:`

To główna sekcja pliku. Zawiera definicje etapów. Wcięcia mają znaczenie: nazwy etapów i ich ustawienia muszą znajdować się na odpowiednich poziomach zagnieżdżenia YAML.

### Etap `prepare`

```yaml
prepare:
  cmd: python src/prepare.py
  deps:
    - src/prepare.py
  outs:
    - data/iris.csv
```

- `prepare` — nazwa etapu.
- `cmd` — polecenie, które DVC uruchamia dla tego etapu.
- `deps` — zależności wejściowe. Tutaj zależnością jest skrypt `src/prepare.py`; jego zmiana może spowodować ponowne wykonanie etapu.
- `outs` — wynik etapu. Skrypt tworzy `data/iris.csv`. DVC śledzi ten plik i domyślnie może przechowywać go w swoim cache.

Skrypt pobiera wbudowany zbiór Iris z scikit-learn i zapisuje go jako CSV. Ten plik staje się wejściem kolejnego etapu.

### Etap `validate`

```yaml
validate:
  cmd: python src/validate.py
  deps:
    - src/validate.py
    - data/iris.csv
  outs:
    - reports/validation.json:
        cache: false
```

- `validate` — nazwa etapu walidacji.
- `cmd` — uruchamia skrypt sprawdzający dane.
- `deps` — etap zależy zarówno od kodu walidacji, jak i od `data/iris.csv`. Zmiana któregokolwiek z nich może wymagać ponownej walidacji.
- `outs` — wynikiem jest `reports/validation.json`, czyli raport informujący, czy dane są poprawne.
- `cache: false` — DVC nie przechowuje tego raportu w swoim cache. Plik nadal jest wynikiem etapu, ale nie jest wersjonowany w cache tak jak zwykłe wyjście.

Skrypt kończy się błędem, jeśli dane nie przejdą walidacji. Dzięki temu następne etapy nie powinny działać na niepoprawnym zbiorze.

### Etap `split`

```yaml
split:
  cmd: python src/split.py
  deps:
    - src/split.py
    - data/iris.csv
    - reports/validation.json
  params:
    - split.test_size
    - split.seed
  outs:
    - data/train.csv
    - data/test.csv
```

- `split` — nazwa etapu dzielącego dane.
- `cmd` — uruchamia `src/split.py`.
- `deps` — skrypt potrzebuje swojego kodu, zbioru Iris i raportu walidacji.
- `params` — etap zależy od dwóch parametrów z `params.yaml`:
  - `split.test_size` — jaka część danych trafi do zbioru testowego;
  - `split.seed` — ziarno losowania, dzięki któremu podział można odtworzyć.
- `outs` — etap tworzy dwa pliki: `data/train.csv` i `data/test.csv`.

Skrypt używa podziału warstwowego (`stratify`), aby zachować proporcje klas w obu częściach danych.

### Etap `train`

```yaml
train:
  cmd: python src/train.py
  deps:
    - src/train.py
    - data/train.csv
  params:
    - split.seed
    - model.max_iter
    - model.C
  outs:
    - models/model.joblib
```

- `train` — nazwa etapu uczenia.
- `cmd` — uruchamia `src/train.py`.
- `deps` — etap zależy od skryptu treningowego oraz danych treningowych.
- `params` — DVC śledzi parametry używane przy trenowaniu:
  - `split.seed` — ziarno przekazywane jako `random_state` modelu;
  - `model.max_iter` — maksymalna liczba iteracji regresji logistycznej;
  - `model.C` — parametr regularyzacji modelu.
- `outs` — wytrenowany model jest zapisywany jako `models/model.joblib`.

Jeśli zmieni się kod treningowy, dane treningowe lub wskazane parametry, DVC może ponownie wykonać ten etap.

### Etap `evaluate`

```yaml
evaluate:
  cmd: python src/evaluate.py
  deps:
    - src/evaluate.py
    - data/test.csv
    - models/model.joblib
  metrics:
    - reports/metrics.json:
        cache: false
```

- `evaluate` — nazwa etapu oceny.
- `cmd` — uruchamia `src/evaluate.py`.
- `deps` — ocena zależy od kodu oceniającego, danych testowych i zapisanego modelu.
- `metrics` — `reports/metrics.json` zawiera wartości przeznaczone do analizy jako metryki.
- `cache: false` — plik metryk nie jest przechowywany w cache DVC. DVC nadal rozpoznaje go jako plik metryk, m.in. na potrzeby `dvc metrics show`.

Skrypt zapisuje `accuracy`, `f1_macro` oraz liczbę obserwacji w zbiorze testowym. Dzięki temu można odczytać wynik modelu bez ponownego analizowania logów programu.

### Parametry z `params.yaml`

Plik `params.yaml` utworzony przez komórkę zawiera:

```yaml
split:
  test_size: 0.3
  seed: 42

model:
  max_iter: 500
  C: 1.0
```

Są to ustawienia używane przez skrypty. DVC zapisuje w definicjach etapów, które konkretnie parametry wpływają na dany etap. Na przykład zmiana `split.test_size` wymaga ponownego podziału danych i może wpłynąć na trenowanie oraz ocenę modelu.

### Zależności całego potoku

W uproszczeniu przepływ wygląda tak:

```text
prepare
   ↓
validate
   ↓
split
  ↙   ↘
train  dane testowe
  ↘   ↙
evaluate
```

`evaluate` potrzebuje jednocześnie modelu z etapu `train` i danych testowych z etapu `split`. DVC wywnioskuje te zależności na podstawie plików zapisanych w `deps` i `outs`.

Ten przykład pokazuje podstawowy sposób opisywania potoku w DVC. W większym projekcie warto też wersjonować pliki `dvc.yaml`, `dvc.lock`, `params.yaml` i kod w Git oraz skonfigurować zdalny magazyn DVC, jeśli dane i modele mają być współdzielone z innymi osobami.

In [ ]:
%%skip True
import os
import re
import subprocess
import textwrap

# `python` w poleceniach etapow ma wskazywac interpreter tego srodowiska.
DVC_ENV = {
    **os.environ,
    "PATH": str(Path(sys.executable).parent) + os.pathsep + os.environ.get("PATH", ""),
}


def write_potok(relative_path: str, content: str) -> Path:
    path = POTOK / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip(), encoding="utf-8")
    return path


def dvc(*arguments, show: bool = True) -> subprocess.CompletedProcess:
    result = subprocess.run(
        [sys.executable, "-m", "dvc", *arguments],
        cwd=POTOK, env=DVC_ENV, capture_output=True, text=True,
    )
    if show:
        text = (result.stdout + result.stderr).strip()
        print(text[-1600:] if text else "(bez komunikatu)")
    return result


def stages_executed(result: subprocess.CompletedProcess) -> list:
    return re.findall(r"Running stage '([^']+)'", result.stdout + result.stderr)


write_potok(
    "params.yaml",
    """
    split:
      test_size: 0.3
      seed: 42

    model:
      max_iter: 500
      C: 1.0
    """,
)

write_potok(
    "src/prepare.py",
    """
    from pathlib import Path

    from sklearn.datasets import load_iris

    Path("data").mkdir(exist_ok=True)
    load_iris(as_frame=True).frame.to_csv("data/iris.csv", index=False)
    print("prepare: data/iris.csv")
    """,
)

write_potok(
    "src/validate.py",
    """
    import json
    import sys
    from pathlib import Path

    import pandas as pd
    import pandera.pandas as pa

    SCHEMA = pa.DataFrameSchema(
        columns={
            "sepal length (cm)": pa.Column(float, pa.Check.in_range(3.0, 9.0)),
            "sepal width (cm)": pa.Column(float, pa.Check.in_range(1.0, 5.0)),
            "petal length (cm)": pa.Column(float, pa.Check.in_range(0.5, 8.0)),
            "petal width (cm)": pa.Column(float, pa.Check.in_range(0.05, 3.0)),
            "target": pa.Column(int, pa.Check.isin([0, 1, 2])),
        },
        strict=True,
    )

    frame = pd.read_csv("data/iris.csv")
    Path("reports").mkdir(exist_ok=True)
    try:
        SCHEMA.validate(frame, lazy=True)
        report = {"valid": True, "rows": len(frame), "violations": 0}
    except pa.errors.SchemaErrors as errors:
        report = {"valid": False, "rows": len(frame), "violations": len(errors.failure_cases)}

    Path("reports/validation.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(f"validate: {report}")
    if not report["valid"]:
        sys.exit(1)
    """,
)

write_potok(
    "src/split.py",
    """
    import json
    from pathlib import Path

    import pandas as pd
    import yaml
    from sklearn.model_selection import train_test_split

    params = yaml.safe_load(Path("params.yaml").read_text(encoding="utf-8"))

    if not json.loads(Path("reports/validation.json").read_text(encoding="utf-8"))["valid"]:
        raise SystemExit("split: dane nie przeszly walidacji")

    frame = pd.read_csv("data/iris.csv")
    train, test = train_test_split(
        frame,
        test_size=params["split"]["test_size"],
        random_state=params["split"]["seed"],
        stratify=frame["target"],
    )
    train.to_csv("data/train.csv", index=False)
    test.to_csv("data/test.csv", index=False)
    print(f"split: {len(train)} / {len(test)}")
    """,
)

write_potok(
    "src/train.py",
    """
    from pathlib import Path

    import joblib
    import pandas as pd
    import yaml
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

    params = yaml.safe_load(Path("params.yaml").read_text(encoding="utf-8"))
    train = pd.read_csv("data/train.csv")
    features = [column for column in train.columns if column != "target"]

    model = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "classifier",
                LogisticRegression(
                    max_iter=params["model"]["max_iter"],
                    C=params["model"]["C"],
                    random_state=params["split"]["seed"],
                ),
            ),
        ]
    ).fit(train[features], train["target"])

    Path("models").mkdir(exist_ok=True)
    joblib.dump(model, "models/model.joblib")
    print("train: models/model.joblib")
    """,
)

write_potok(
    "src/evaluate.py",
    """
    import json
    from pathlib import Path

    import joblib
    import pandas as pd
    from sklearn.metrics import accuracy_score, f1_score

    test = pd.read_csv("data/test.csv")
    features = [column for column in test.columns if column != "target"]
    predictions = joblib.load("models/model.joblib").predict(test[features])

    metrics = {
        "accuracy": round(float(accuracy_score(test["target"], predictions)), 6),
        "f1_macro": round(float(f1_score(test["target"], predictions, average="macro")), 6),
        "n_test": int(len(test)),
    }
    Path("reports").mkdir(exist_ok=True)
    Path("reports/metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    print(f"evaluate: {metrics}")
    """,
)

write_potok(
    "dvc.yaml",
    """
    stages:
      prepare:
        cmd: python src/prepare.py
        deps:
          - src/prepare.py
        outs:
          - data/iris.csv

      validate:
        cmd: python src/validate.py
        deps:
          - src/validate.py
          - data/iris.csv
        outs:
          - reports/validation.json:
              cache: false

      split:
        cmd: python src/split.py
        deps:
          - src/split.py
          - data/iris.csv
          - reports/validation.json
        params:
          - split.test_size
          - split.seed
        outs:
          - data/train.csv
          - data/test.csv

      train:
        cmd: python src/train.py
        deps:
          - src/train.py
          - data/train.csv
        params:
          - split.seed
          - model.max_iter
          - model.C
        outs:
          - models/model.joblib

      evaluate:
        cmd: python src/evaluate.py
        deps:
          - src/evaluate.py
          - data/test.csv
          - models/model.joblib
        metrics:
          - reports/metrics.json:
              cache: false
    """,
)

if not (POTOK / ".dvc").exists():
    dvc("init", "--no-scm", "-q", show=False)

print("Projekt potoku gotowy:")
for path in sorted(POTOK.rglob("*")):
    if path.is_file() and ".dvc" not in path.parts:
        print(f"  {path.relative_to(POTOK).as_posix()}")

## 13. Pierwsze uruchomienie

`dvc repro` czyta graf, ustala kolejność i wykonuje etapy. Nie musimy nikomu mówić, że `train` ma się wykonać po `split` — wynika to z zadeklarowanych zależności.

**Do opisania:** w jakiej kolejności wykonały się etapy i skąd DVC wziął tę kolejność?

In [ ]:
first_run = dvc("repro")
print(f"\nWykonane etapy: {stages_executed(first_run)}")
print(f"Powstał dvc.lock: {(POTOK / 'dvc.lock').exists()}")

## 14. Punkt kontrolny 4 — powtórne uruchomienie bez zmian

Uruchamiamy to samo polecenie drugi raz. Nic się nie zmieniło: ani kod, ani dane, ani parametry.

**Do opisania:** ile etapów wykonało się ponownie i na jakiej podstawie DVC podjął tę decyzję?

In [ ]:
second_run = dvc("repro")
print(f"\nWykonane etapy: {stages_executed(second_run) or 'żaden'}")
print(
    "\nDVC porównuje sumy kontrolne zapisane w dvc.lock z aktualnym stanem plików.\n"
    "Znacznik czasu nie ma tu znaczenia — liczy się zawartość."
)

## 15. Punkt kontrolny 5 — zmiana parametru

Zmieniamy pojedynczy parametr w `params.yaml` i sprawdzamy, jak daleko sięga unieważnienie.

Najpierw `model.max_iter`, który dotyczy tylko treningu. Potem `split.test_size`, który zmienia podział danych, a więc wszystko poniżej.

**Do opisania:** dlaczego etap `prepare` nie wykonał się w żadnym z tych przypadków?

In [ ]:
PARAMS_PATH = POTOK / "params.yaml"
original_params = PARAMS_PATH.read_text(encoding="utf-8")

try:
    for description, old_text, new_text in (
        ("model.max_iter: 500 -> 800", "max_iter: 500", "max_iter: 800"),
        ("split.test_size: 0.3 -> 0.25", "test_size: 0.3", "test_size: 0.25"),
    ):
        current_params = PARAMS_PATH.read_text(encoding="utf-8")
        if current_params.count(old_text) != 1:
            raise ValueError(f"Oczekiwano dokładnie jednego wystąpienia {old_text!r}.")
        PARAMS_PATH.write_text(current_params.replace(old_text, new_text, 1), encoding="utf-8")
        result = dvc("repro", show=False)
        output = result.stdout + result.stderr
        if result.returncode != 0:
            raise RuntimeError(f"dvc repro nie powiódł się:\n{output}")

        cached_stages = re.findall(r"Stage '([^']+)' is cached - skipping run", output)
        actually_run = stages_executed(result)
        affected_stages = list(dict.fromkeys(actually_run + cached_stages))
        print(description)
        print(f"    etapy wymagające aktualizacji: {affected_stages}")
        print(f"    uruchomione ponownie: {actually_run}")
        print(f"    odtworzone z cache: {cached_stages}\n")
finally:
    PARAMS_PATH.write_text(original_params, encoding="utf-8")
    restore_result = dvc("repro", show=False)
    if restore_result.returncode != 0:
        raise RuntimeError(
            "Nie udało się odtworzyć potoku po przywróceniu parametrów:\n"
            f"{restore_result.stdout}{restore_result.stderr}"
        )
    print("Przywrócono pierwotne parametry.")

## 16. Graf zależności i metryki

Dwie rzeczy, które dostajemy za darmo, gdy potok jest opisany deklaratywnie: rysunek grafu i zestawienie metryk. Obie biorą się z tego samego pliku, więc nie mogą rozjechać się z rzeczywistością.

Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
graph = dvc("dag", "--md", show=False)
print(graph.stdout.strip() or graph.stderr.strip()[-400:])

metrics = dvc("metrics", "show", "--json", show=False)
print("\nMetryki potoku:")
print(metrics.stdout.strip() or metrics.stderr.strip()[-400:])

print(
    "\nPlik dvc.lock zapisuje sumy kontrolne wszystkich wejść i wyjść każdego etapu.\n"
    "To on trafia do repozytorium razem z dvc.yaml i to on odpowiada na pytanie,\n"
    "z jakich dokładnie danych i parametrów powstał ten model."
)

## 17. Podsumowanie

Cechy przestały być zbiorem operacji rozrzuconych po notebooku. Istnieje jedna transformacja, test chroniący przed rozjazdem ścieżek oraz potok opisany w `dvc.yaml`, który sam ustala kolejność etapów i przelicza wyłącznie to, co faktycznie się zmieniło.

Czego nadal brakuje:

- kolejne przebiegi nadpisują wyniki, więc nie da się porównać eksperymentów (zajęcia 5);
- test zgodności istnieje tylko w notebooku, a nie jako automatycznie uruchamiany zestaw (zajęcia 6);
- model nie ma sygnatury ani wersji, więc nie da się go bezpiecznie udostępnić (zajęcia 7–8).

### Zadanie do własnego projektu

1. Jedna funkcja lub obiekt budujący cechy, używany w treningu i w predykcji.
2. Test zgodności obu ścieżek dla tych samych danych wejściowych.
3. Sprawdzenie, czy w potoku nie ma wycieku przez przetwarzanie.
4. Podział czasowy, jeżeli problem ma wymiar czasu, wraz z opisem zmiany metryk.
5. Potok opisany w `dvc.yaml` z zadeklarowanymi `deps`, `outs`, `params` i `metrics`.
6. Walidacja danych jako etap, od którego zależą etapy dalsze — nie jako osobny rytuał.
7. Wykazane działanie pamięci podręcznej: powtórne `dvc repro` bez zmian nie przelicza niczego.
8. Wykazany zasięg unieważnienia: zmiana jednego parametru przelicza tylko etapy zależne.
9. Plik `dvc.lock` w repozytorium.
10. Odpowiedź w dokumentacji: czy projekt potrzebuje feature store i dlaczego.